---
## <font color="red">1 | SYSTÈME DE</font> RAISONNEMENT
---

Dans ce notebook, vous apprendrez à implémenter un système de raisonnement avec des connaissances de base sur le monde des meurtres.

### <font color="red">1.1 - Les</font> paquets
---
Importons d’abord tous les packages dont vous aurez besoin. Si dans votre environnement python il vous manque `nltk`, vous pouvez l'installer avec la commande `pip intall <paquet>` :

* [nltk](https://www.nltk.org/) : C'est une bibliothèque permettant le traitement automatique des langues.
* [aima-python](https://github.com/aimacode/aima-python) : Code Python pour le livre Intelligence artificielle: une approche moderne.

Pour le paquet `aima-python`, vous devez le télécharger et copier tous les fichiers .py dans le dossier de votre projet.

In [226]:
from aima.logic import *
import nltk

### <font color="red">1.2 - Le moteur d'</font> inférence
---
Ensuite, une classe est créée, qui est le moteur d'inférence. Cette classe contient des listes de personnes, d'armes et de pièces existant dans le monde.

Le moteur d'inférence contient une liste des `clauses` (faits) survenues et avec lesquelles la base de connaissances `crime_kb` est alimentée.

Notez que la base de connaissances est un objet de la classe `FolKB` (First-order logic knowled base) du package `aima` à laquelle vous pouvez dire (`tell`) et demander (`ask`) des phrases.

Nous pouvons communiquer des faits à la base de connaissances avec la fonction `add_clause` qui utilise la fonction `tell` du paquet `aima`.

Enfin, nous pouvons demander des informations à la base de connaissances avec la fonction `ask` du paquet `aima`, utilisée dans les fonctions `get_victim`, `get_crime_room`, `get_crime_weapon`, `get_crime_hour`, `get_suspect` et `get_innocent`.

Il est important d'indiquer que les fonctions `tell` et `ask` reçoivent des expressions logiques du premier ordre en tant que paramètres.

In [ ]:
# Permet d'inferer qui est le meurtrier, quand, comment, où il a tué.
class CrimeInference:

    def __init__(self):
        self.weapons = ["Marteau", "Collier", "Chandelier", "Hache", "Corde", "Fusil", "Couteau"]
        self.rooms = ["Cuisine", "Bureau", "Garage", "Salon", "Bibliotheque", "Jardin", "Chambre"]
        self.persons = ["Moutarde", "Peacock", "Scarlett", "Plum", "White", "Black", "Green", "Violet"]
        
        # Liste de clauses (faits) qui seront stockées dans la base de connaissance.
        self.clauses = []        
        
        self.base_clauses()
        self.initialize_KB()
        self.inference_rules()
        
        # Base de connaissances (First-order logic - FOL)
        self.crime_kb = FolKB(self.clauses)

    # Déclaration dans la logique du premier ordre
    def base_clauses(self):
        # Le paramètre est une arme
        self.arme_clause = 'Arme({})'
        
        # Le paramètre est une pièce
        self.piece_clause = 'Piece({})'
        
        # Le paramètre est une persone
        self.personne_clause = 'Personne({})'

        # paramètre 1 : arme; paramètre 2 : pièce
        # p.ex.: Le couteau se trouve dans la cuisine
        self.weapon_room_clause = 'Arme_Piece({},{})'

        # paramètre 1 : personne; paramètre 2 : pièce; paramètre 3 : heure
        # p.ex.: Mustart était dans la cuisine à 11h00
        self.person_room_hour_clause = 'Personne_Piece_Heure({}, {}, {})'

        # paramètre 1 : personne; paramètre 2 : piece
        # p.ex.: Mustard se trouve dans la cuisine
        self.person_room_clause = 'Personne_Piece({}, {})'

        # paramète 1 : personne
        # p. ex.: Mustard est mort
        self.dead_clause = 'EstMort({})'
        
        # paramète 1 : personne
        # p. ex.: Mustard est vivant
        self.alive_clause = 'EstVivant({})'

        # paramètre 1 : personne
        # p. ex.: Mustard est la victime
        self.victim_clause = 'Victime({})'

        # paramètre 1 : piece; paramètre 2 : piece
        self.room_different_clause = 'PieceDifferente({},{})'

        # paramètre 1 : piece; paramètre 2 : piece
        self.weapon_different_clause = 'ArmeDifferente({},{})'

        # paramètre 1 : personne; paramètre 2 : personne
        self.person_different_clause = 'PersonneDifferente({},{})'

        # paramètre 1 : heure
        self.crime_hour_clause = 'HeureCrime({})'

        # paramètre 1 : heure
        self.crime_hour_plus_one_clause = 'UneHeureApresCrime({})'

        # paramètre 1 : personne
        # p. ex.: Mustard a des marques au cou
        self.body_mark_clause = 'MarqueCou({})'

        # paramètre 1 : personne
        self.split_skull_clause = 'CraneFendu({})'

        # paramètre 1 : personne
        self.chest_wound_clause = 'PlaiePoitrine({})'

        # paramètre 1 : personne
        self.cut_leg_clause = 'JambeCoupee({})'

        # paramètre 1 : personne
        self.broken_arm_clause = 'BrasCasse({})'

        # paramètre 1 : personne
        self.chest_hole_clause = 'TrouPoitrine({})'

        # paramètre 1 : personne
        self.intoxication_clause = 'Intoxication({})'

        self.weapon_room_hour_clause = 'Arme_Piece_Heure({}, {}, {})'

    def initialize_KB(self):
        # Clause pour differencier les pièces
        for i in range(len(self.rooms)):
            for j in range(len(self.rooms)):
                if i != j:
                    # Le bureau est different de la cuisine = PieceDifferente(Bureau, Cuisine)
                    self.clauses.append(expr(self.room_different_clause.format(self.rooms[i], self.rooms[j])))

        # Clause pour differencier les armes
        for i in range(len(self.weapons)):
            for j in range(len(self.weapons)):
                if i != j:
                    # Le couteau est different de la corde = ArmeDifferente(Couteau, Corde)
                    self.clauses.append(expr(self.weapon_different_clause.format(self.weapons[i], self.weapons[j])))

        #Clause pour differencier les personnes
        for i in range(len(self.persons)):
            for j in range(len(self.persons)):
                if i != j:
                    # Mustard est different de Scarlett = PersonneDifferente(Mustard, Scarlett)
                    self.clauses.append(expr(self.person_different_clause.format(self.persons[i], self.persons[j])))

        # Initialiser KB sur Armes, Pieces, Personnes
        for weapon in self.weapons:
            # Le couteau est une arme = Arme(Couteau)
            self.clauses.append(expr(self.arme_clause.format(weapon)))

        for room in self.rooms:
            # La cuisine est une pièce = Piece(Cuisine)
            self.clauses.append(expr(self.piece_clause.format(room)))

        for person in self.persons:
            # Mustar est une personne = Personne(Mustard)
            self.clauses.append(expr(self.personne_clause.format(person)))
    
    # Expressions dans la logique du premier ordre permettant de déduire les caractéristiques du meurtre
    def inference_rules(self):
        # Determine la piece du crime
        self.clauses.append(expr('EstMort(x) & Personne_Piece(x, y) ==> PieceCrime(y)'))

        # Determiner l'arme potentielle du crime
        self.clauses.append(expr('PlaiePoitrine(x) & CauseBlessure(a, PlaiePoitrine) ==> ArmePossible(a)'))
        self.clauses.append(expr('CraneFendu(x) & CauseBlessure(a, CraneFendu) ==> ArmePossible(a)'))
        self.clauses.append(expr('MarqueCou(x) & CauseBlessure(a, MarqueCou) ==> ArmePossible(a)'))


        #Determiner arme crime ()
        self.clauses.append(expr('ArmePossible(a) & PieceCrime(r) & HeureCrime(h) & Arme_Piece_Heure(a, r, h) ==> ArmeCrime(a)'))


        # Si la personne est morte alors elle est la victime et ce n'est pas un suicide
        self.clauses.append(expr('EstMort(x) ==> Victime(x)'))

        # Si la personne est morte alors elle est innocente et ce n'est pas un suicide
        self.clauses.append(expr('EstMort(x) ==> Innocent(x)'))

        # Personne suspect (Prend en considération que le meutrier n'est pas rester dans la même pièce que la victime une heure après le crime)
        self.clauses.append(expr('EstVivant(p) & PieceCrime(r1) & UneHeureApresCrime(h1) & Personne_Piece_Heure(p, r2, h1) & ArmeCrime(a) & Arme_Piece(a, r2) & PieceDifferente(r1, r2) ==> Suspect(p)'))

        #Personnes innocente
        self.clauses.append(expr('Suspect(x) & EstVivant(y) & PersonneDifferente(x,y) ==> Innocent(y)'))

    # Ajouter des clauses, c'est-à-dire des faits, à la base de connaissances
    def add_clause(self, clause_string):
        self.crime_kb.tell(expr(clause_string))

    # Demander à la base de connaissances qui est la victime
    def get_victim(self):
        result = self.crime_kb.ask(expr('Victime(x)'))
        if not result:
            return False
        else:
            return result[x]
        
    # Demander à la base de connaissances la pièce du meurtre
    def get_crime_room(self):
        result = self.crime_kb.ask(expr('PieceCrime(x)'))
        if not result:
            return False
        else:
            return result[x]

    # Demander à la base de connaissances l'arme du meurtrier
    def get_crime_weapon(self):
        result = self.crime_kb.ask(expr('ArmeCrime(x)'))
        if not result:
            return result
        else:
            return result[x]

    # Demander à la base de connaissances l'heure du meurtre
    def get_crime_hour(self):
        result = self.crime_kb.ask(expr('HeureCrime(x)'))
        if not result:
            return result
        else:
            return result[x]

    def get_crime_hour_plus_one(self):
        result = self.crime_kb.ask(expr('UneHeureApresCrime(x)'))
        if not result:
            return result
        else:
            return result[x]
    
    # Demander à la base de connaissances le suspect
    def get_suspect(self):
        result = self.crime_kb.ask(expr('Suspect(x)'))
        if not result:
            return result
        else:
            return result[x]

    # Demander à la base de connaissances la liste d'innocents
    def get_innocent(self):
        result = list(fol_bc_ask(self.crime_kb, expr('Innocent(x)')))
        res = []

        for elt in result:
            if not res.__contains__(elt[x]):
                res.append(elt[x])
        return res

### <font color="red">1.3 - Les faits et </font> la déduction
---
Maintenant que nous avons le moteur de déduction, il est temps de commencer à alimenter la base de connaissances avec les faits du monde.

Comme nous communiquerons les faits au moteur d’inférence en français, plusieurs grammaires ont été créées pour interpréter les phrases. Celles-ci se trouvent dans le dossier `grammars`.

À partir des grammaires, nous pouvons analyser les phrases et puis les transformer en expressions logiques du premier ordre. C'est à cela que servent les fonctions `results_as_string` et `to_fol`. 

In [287]:
# Cette fonction retourne le format d'une expression logique de premier ordre
def results_as_string(results):
    res = ''
    for result in results:
        # synrep = syntactic representation
        # semrep = semantic representation
        for (synrep, semrep) in result:            
            res += str(semrep)
    return res

La fonction `nltk.interpret_sents` donne une représentation de premier ordre d'un fait en fraçais à partir d'une grammaire. La représentation est récupérée et transmise à la fonction `results_as_string` pour la formater sous la forme Exp(x). Enfin, l'expression mise en forme est renvoyée pour l'ajouter à la base de connaissances.

In [288]:
# Cette fonction transforme une phrase en fraçais dans une expression logique du premier ordre
def to_fol(fact, grammar):
    grammar_path = grammar if grammar.startswith('grammars/') else f'grammars/{grammar}'
    with open(grammar_path, 'r', encoding='utf-8') as f:
        grammar_obj = nltk.grammar.FeatureGrammar.fromstring(f.read())
    sent = results_as_string(nltk.interpret_sents(fact, grammar_obj))
    print(sent)
    return sent

Nous créons maintenant une instance du moteur d'inférence et commençons à ajouter les faits à la base de connaissances. Les faits sont communiqués en français, interprétés par la grammaire appropriée et ajoutés.

In [289]:
agent = CrimeInference()

In [290]:
#Faits
facts = [['Black est mort'],
         ['Moutarde est vivant'],
         ['Plum est vivant'],
         ['Green est vivant'],
         ['Scarlett est vivant'],
         ['Violet est vivant'],
         ['White est vivant'],
         ['Peacock est vivant']]

#Les fait sont ajoutés à la base de connaissances
agent.add_clause(to_fol(facts[0], 'grammars/personne_morte.fcfg'))
facts.pop(0)

for fact in facts:    
    agent.add_clause(to_fol(fact, 'grammars/personne_vivant.fcfg'))

EstMort(Black)
EstVivant(Moutarde)
EstVivant(Plum)
EstVivant(Green)
EstVivant(Scarlett)
EstVivant(Violet)
EstVivant(White)
EstVivant(Peacock)


In [291]:
facts = [
    "Un fusil cause une plaie à la poitrine",
    "Un couteau cause une plaie à la poitrine",
    "Un marteau cause un crâne fendu",
    "Une hache cause un crâne fendu",
    "Un chandelier cause un crâne fendu",
    "Une corde cause des marques au cou",
    "Un collier cause des marques au cou"
]

for f in facts:
    agent.add_clause(to_fol([f], 'grammars/arme_blessure.fcfg'))

CauseBlessure(Fusil,PlaiePoitrine)
CauseBlessure(Couteau,PlaiePoitrine)
CauseBlessure(Marteau,CraneFendu)
CauseBlessure(Hache,CraneFendu)
CauseBlessure(Chandelier,CraneFendu)
CauseBlessure(Corde,MarqueCou)
CauseBlessure(Collier,MarqueCou)


In [ ]:
#Dans le garage

# On se rend compte que Black est morte par étranglement dans le garage
fact = ['Black a un crâne fendu']
agent.add_clause(to_fol(fact, 'grammars/personne_crane.fcfg'))

print(agent.get_crime_weapon())

fact = ['Black est dans le garage']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# On sait que Peacock est dans le garage
fact = ['Violet est dans le garage']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

fact = ['La hache est dans le garage']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))


# 1 Question : À quelle heure Black est mort ? l'heure du decès -> Rep : 16h
fact = ['Black est mort à 16h']
agent.add_clause(to_fol(fact, 'grammars/personne_morte_heure.fcfg'))

uneHeureApres = agent.get_crime_hour() + 1

agent.add_clause('UneHeureApresCrime({})'.format(uneHeureApres))

# 2 Question : Ou se trouvait Violet à 17h ? -> Rep : Violet dans le Garage à 17h
fact = ['Violet était dans le garage à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))


# 3 Question : Où était la hache à 16h ? -> Rep : La hache était dans le garage à 16h
fact = ['La hache était dans le garage à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))


CraneFendu(Black)
False
Personne_Piece(Black,Garage)
Personne_Piece(Violet,Garage)
Arme_Piece(Hache,Garage)
HeureCrime(16)
Personne_Piece_Heure(Violet,Garage,17)
Arme_Piece_Heure(Hache,Garage,16)


In [293]:
# Dans le salon

# Voit qu'il y a un marteau et Moutarde dans le salon
fact = ['Le marteau est dans le salon']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['Moutarde est dans le salon']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 4 Question : Ou se trouvait Moutarde à 17h ? -> Rep : Moutarde dans le Jardin à 17h
fact = ['Moutarde était dans le jardin à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 5 Question : Où était la marteau à 16h ? -> Rep : La marteau était dans le Garage à 16h
fact = ['Le marteau était dans le garage à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))


Arme_Piece(Marteau,Salon)
Personne_Piece(Moutarde,Salon)
Personne_Piece_Heure(Moutarde,Jardin,17)
Arme_Piece_Heure(Marteau,Garage,16)


In [294]:
# Dans la cuisine

# Voit qu'il y a un couteau et Plum dans la cuisine
fact = ['Le couteau est dans la cuisine']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['Plum est dans la cuisine']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 6 Question : Ou se trouvait Plum à 17h ? -> Rep : Plum était dans la bibliothèque à 17h
fact = ['Plum était dans la bibliotheque à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 7 Question : Où était la couteau à 16h ? -> Rep : La couteau était dans le Bureau à 16h
fact = ['Le couteau était dans le bureau à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))

Arme_Piece(Couteau,Cuisine)
Personne_Piece(Plum,Cuisine)
Personne_Piece_Heure(Plum,Bibliotheque,17)
Arme_Piece_Heure(Couteau,Bureau,16)


In [295]:
# Dans la bibliothèque

# Voit qu'il y a une corde et Scarlett dans la bibliothèque
fact = ['La corde est dans la bibliotheque']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['Scarlett est dans la bibliotheque']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 8 Question : Ou se trouvait Scarlett à 17h ? -> Rep : Scarlett était dans le bureau à 17h
fact = ['Scarlett était dans le bureau à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 9 Question : Où était la corde à 16h ? -> Rep : La corde était dans le Jardin à 16h
fact = ['La corde était dans le jardin à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))

Arme_Piece(Corde,Bibliotheque)
Personne_Piece(Scarlett,Bibliotheque)
Personne_Piece_Heure(Scarlett,Bureau,17)
Arme_Piece_Heure(Corde,Jardin,16)


In [296]:
# Dans le bureau

# Voit qu'il y a un collier, White
fact = ['Le collier est dans le bureau']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['White est dans le bureau']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 10 Question : Ou se trouvait White à 17h ? -> Rep : White était dans le Salon à 17h
fact = ['White était dans le salon à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 11 Question : Où était la collier à 16h ? -> Rep : La collier était dans la Cuisine à 16h
fact = ['Le collier était dans la cuisine à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))




Arme_Piece(Collier,Bureau)
Personne_Piece(White,Bureau)
Personne_Piece_Heure(White,Salon,17)
Arme_Piece_Heure(Collier,Cuisine,16)


In [297]:
# Dans le jardin

# Voit qu'il y a un fusil, Peacock dans le jardin 
fact = ['Le fusil est dans le jardin']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['Peacock est dans le jardin']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 12 Question : Ou se trouvait Peacock à 17h ? -> Rep : Peacock était dans la chambre à 17h
fact = ['Peacock était dans la chambre à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 13 Question : Où était la fusil à 16h ? -> Rep : La fusil était dans la Bibliothèque à 16h
fact = ['Le fusil était dans la bibliotheque à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))



Arme_Piece(Fusil,Jardin)
Personne_Piece(Peacock,Jardin)
Personne_Piece_Heure(Peacock,Chambre,17)
Arme_Piece_Heure(Fusil,Bibliotheque,16)


In [298]:
# Dans la chambre

# Voit qu'il y a un chandelier, Green dans la chambre
fact = ['Le chandelier est dans la chambre']
agent.add_clause(to_fol(fact, 'grammars/arme_piece.fcfg'))

fact = ['Green est dans la chambre']
agent.add_clause(to_fol(fact, 'grammars/personne_piece.fcfg'))

# 14 Question : Ou se trouvait Green à 17h ? -> Rep : Green était dans la Cuisine à 17h
fact = ['Green était dans la cuisine à ' + str(uneHeureApres) + 'h']
agent.add_clause(to_fol(fact, 'grammars/personne_piece_heure.fcfg'))

# 15 Question : Où était la chandelier à 16h ? -> Rep : La chandelier était dans la chambre à 16h
fact = ['Le chandelier était dans la chambre à 16h']
agent.add_clause(to_fol(fact, 'grammars/arme_piece_heure.fcfg'))



Arme_Piece(Chandelier,Chambre)
Personne_Piece(Green,Chambre)
Personne_Piece_Heure(Green,Cuisine,17)
Arme_Piece_Heure(Chandelier,Chambre,16)


Enfin, après avoir fourni tous les faits (scénario), nous demandons au moteur d’inférence de tirer ses conclusions.

In [299]:
# Conclusions
print("Pièce du crime : ", agent.get_crime_room())
print("Arme du crime : ", agent.get_crime_weapon())
print("Personne victime : ", agent.get_victim())
print("Heure du crime : ", agent.get_crime_hour())
print("Meurtrier : ", agent.get_suspect())
print("Personnes innocentes : ", agent.get_innocent())

Pièce du crime :  Garage
Arme du crime :  Marteau
Personne victime :  Black
Heure du crime :  16
Meurtrier :  Violet
Personnes innocentes :  [Black, Moutarde, Plum, Green, Scarlett, White, Peacock, Violet]
